In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.utils import to_categorical


data = pd.read_csv('Graph metrics.csv')  


X = data.drop(['Class', 'SUBID'], axis=1).values  
y = data['Class'].values


scaler = MinMaxScaler()
X = scaler.fit_transform(X)

def calculate_specificity(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    specificity = tn / (tn + fp)
    return specificity

def create_1d_cnn(input_shape):
    model = Sequential([
        Conv1D(32, 3, activation='relu', padding='same', input_shape=input_shape),
        BatchNormalization(),
        MaxPooling1D(pool_size=2),
        Dropout(0.2),

        Conv1D(64, 3, activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling1D(pool_size=2),
        Dropout(0.2),

        Flatten(),
        Dense(32, activation='relu'),
        Dropout(0.2),
        Dense(2, activation='softmax')
    ])

    model.compile(optimizer=Adam(learning_rate=0.0005),  # Slower learning rate
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    return model


from tensorflow.keras.callbacks import EarlyStopping
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)



# Feature subset sizes
top_features = [5, 10, 20, 40, 50, 75, 100, 125, 150, 175, 200, 225, 250, 275, 300, 325, 350, 375, 400, 425, 450]

# Cross-validation
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = []

for num_features in top_features:
    print(f"Training with top {num_features} features...")
    X_selected = X[:, :num_features]  # Select top N features
    X_selected = X_selected.reshape(X_selected.shape[0], X_selected.shape[1], 1)  # Reshape for CNN
    y_categorical = to_categorical(y)
    
    fold_results = []
    
    for fold, (train_idx, test_idx) in enumerate(kf.split(X_selected, y)):
        X_train, X_test = X_selected[train_idx], X_selected[test_idx]
        y_train, y_test = y_categorical[train_idx], y_categorical[test_idx]
        
        model = create_1d_cnn(X_train.shape[1:])
        model.fit(X_train, y_train, epochs=30, batch_size=16, validation_data=(X_test, y_test), verbose=0)
        
        y_pred = model.predict(X_test)
        y_pred_classes = np.argmax(y_pred, axis=1)
        y_true = np.argmax(y_test, axis=1)
        
        accuracy = accuracy_score(y_true, y_pred_classes)
        precision = precision_score(y_true, y_pred_classes, average='weighted')
        recall = recall_score(y_true, y_pred_classes, average='weighted')
        f1 = f1_score(y_true, y_pred_classes, average='weighted')
        auc = roc_auc_score(y_test, y_pred, multi_class='ovr')
        specificity = calculate_specificity(y_true, y_pred_classes)
        
        fold_results.append([accuracy, precision, recall, f1, auc, specificity])
    
    # Compute mean across folds
    mean_results = np.mean(fold_results, axis=0)
    results.append([num_features] + list(mean_results))

# Save results
df_results = pd.DataFrame(results, columns=['Top Features', 'Accuracy', 'Precision', 'Recall', 'F1 Score', 'AUC', 'Specificity'])
df_results.to_excel('cnn_classification.xlsx', index=False)

